# ─────────────────────────────────────────────────────
# Monitoramento da Condição | LDC_SKF
## Notebook de Testes
### **Referência:** Código Python dashboard streamlit
### Cada função abaixo, replica uma função do app;
### Basear-se nas linhas originais referenciadas
# ─────────────────────────────────────────────────────

----
1. [Setup — Imports, Paleta, Config](#1-setup)
2. [Autenticação e Token](#2-autenticação)
3. [Assets — get_assets + build_asset_index](#3-assets)
4. [Points — get_points / get_points_v1](#4-points)
5. [Tendência — get_trend + parse_trend](#5-tendência)
6. [Espectro FFT — get_spectrum + parse_spectrum](#6-espectro)
7. [IMx-1 Comissionamento — Varredura Completa](#7-imx-1)
8. [Fleet — Gateways, Sensores, Dispositivos](#8-fleet)
9. [build_asset_index — Teste do Parser de Path](#9-path-parser)

# ─────────────────────────────────────────────────────
# 1. SETUP
# ─────────────────────────────────────────────────────

In [7]:
# ── Imports (espelho das linhas 7-13 do app) ─────────────────
import requests
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timezone, timedelta
import time
import json
import re

print("✅ Imports OK")

✅ Imports OK


In [8]:
# ── Paleta LDC (variáveis CSS → constantes Python) ───────────
# Referência: linhas 32-57 do app (bloco :root)
C_BG        = "#0e1820"   # --bg-base
C_PANEL     = "#162130"   # --bg-card
C_INNER     = "#1c2b3a"   # --bg-panel
C_BORDER    = "#2a3f52"   # --border
C_ACCENT    = "#A7C5E2"   # --accent      (LDC Blue claro)
C_SOLID     = "#32556E"   # --accent-solid (LDC Blue)
C_DEEP      = "#007CAA"   # --accent-deep
C_OK        = "#4E9D2D"   # --ok          (LDC Green)
C_WARN      = "#BA944B"   # --warn        (Âmbar)
C_DANGER    = "#F06A22"   # --danger      (Laranja crítico)
C_TEAL      = "#379A8D"   # --teal
C_TEAL_LT   = "#98C0B8"   # --teal-light
C_PURPLE    = "#5E699E"   # --purple
C_TEXT      = "#EEF4F9"   # --text-hi
C_TEXT_MID  = "#98C0B8"   # --text-mid
C_GRID      = "#2a3f52"   # grids Plotly

# Layout Plotly base — reutilizar em todos os plots
PLOTLY_BASE = dict(
    paper_bgcolor=C_BG,
    plot_bgcolor=C_PANEL,
    font=dict(family="sans-serif", color=C_TEXT_MID, size=11),
    xaxis=dict(gridcolor=C_GRID, zeroline=False,
               tickfont=dict(size=10, color=C_TEXT_MID)),
    yaxis=dict(gridcolor=C_GRID, zeroline=False,
               tickfont=dict(size=10, color=C_TEXT_MID)),
    legend=dict(bgcolor="rgba(22,33,48,0.9)", bordercolor=C_BORDER,
                borderwidth=1, font=dict(size=10, color=C_ACCENT)),
    margin=dict(l=60, r=40, t=60, b=50),
    hovermode="x unified",
)

print("✅ Paleta LDC definida")

✅ Paleta LDC definida


In [9]:
# ── Configuração de conexão ────────────────────────────────────
UNITS = {
    "Alto Araguaia":      "http://services.repcenter.skf.com:21221",
    "Itumbiara":          "http://services.repcenter.skf.com:21236",
    "Jataí":              "http://services.repcenter.skf.com:21226",
    "Paraguaçu Paulista": "http://services.repcenter.skf.com:21246",
    "Ponta Grossa":       "http://services.repcenter.skf.com:21241",
}

UNIT_NAME = "Alto Araguaia"      # ← altere aqui
BASE_URL  = UNITS[UNIT_NAME]
USERNAME  = "patrick.coelho"
PASSWORD  = "R_z]6T]d?8H#v?3M?ja5N?o~5"                   # ← preencha antes de executar

TIMEOUT = 30

In [10]:
# Constantes IMx-1 (linhas 731-737 do app)
IMX1_NODE_TYPES     = {11101, 11102, 11103, 11104}
IMX1_TEMP_NODE_TYPE = 11104
IMX1_TEMP_EU_TYPE   = 10905
IMX1_FROM_DATE      = "2024-05-01T00:00:00"

In [11]:
# Códigos Fleet (linhas 1059-1070 do app)
SYNC_STATUS = {
    0:   ("Não Sincronizado", "warn"),
    1:   ("Sincronizado",     "ok"),
    2:   ("Pendente",         "warn"),
    100: ("Falha",            "danger"),
}
DIAG_BITS = {1: "Bateria Baixa", 512: "Instabilidade de Rede"}
CONN_STATE_LABELS = {
    0: "Desconectado",
    1: "Conectado",
    2: "Sem Medição",
    3: "Conectado — Sem Medição",
}

print(f"Unidade : {UNIT_NAME}")
print(f"URL     : {BASE_URL}")

Unidade : Alto Araguaia
URL     : http://services.repcenter.skf.com:21221


# ─────────────────────────────────────────────────────────────
# 2. AUTENTICAÇÃO
# ─────────────────────────────────────────────────────────────

In [12]:
# ── autenticar() — linha 273 ─────────────────────────────────
def autenticar(base_url, username, password):
    resp = requests.post(
        f"{base_url}/token",
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={"grant_type": "password", "username": username, "password": password},
        timeout=TIMEOUT,
    )
    resp.raise_for_status()
    return resp.json()["access_token"]

TOKEN      = autenticar(BASE_URL, USERNAME, PASSWORD)
TOKEN_TIME = time.time()
print(f"✅ Token: {TOKEN[:40]}…")

✅ Token: eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJ…


In [13]:
# ── ensure_token() standalone (sem st.session_state) ─────────
# Versão adaptada da linha 740 — usa variáveis locais em vez de session_state
_token_cache = {"token": TOKEN, "ts": TOKEN_TIME}

def ensure_token():
    if time.time() - _token_cache["ts"] < 18 * 60:
        return _token_cache["token"]
    print("🔄 Renovando token…")
    tok = autenticar(BASE_URL, USERNAME, PASSWORD)
    _token_cache.update({"token": tok, "ts": time.time()})
    return tok

def hdrs():
    return {"Authorization": f"Bearer {ensure_token()}",
            "Accept": "application/json"}

print("✅ ensure_token() pronto")

✅ ensure_token() pronto


# ─────────────────────────────────────────────────────────────
# 3. ASSETS + build_asset_index
# ─────────────────────────────────────────────────────────────

In [14]:
# ── get_assets() — linha 283 ─────────────────────────────────
def get_assets():
    resp = requests.get(
        f"{BASE_URL}/v2/assets",
        headers=hdrs(),
        params={"includeAcknowledged": "true"},
        timeout=TIMEOUT,
    )
    resp.raise_for_status()
    data = resp.json()
    return data if isinstance(data, list) else data.get("value", [])

assets = get_assets()
print(f"✅ {len(assets)} asset(s)")

# Inspeciona o primeiro para ver os campos disponíveis
print("\\nCampos do 1º asset:")
for k, v in list(assets[0].items())[:15]:
    print(f"  {k}: {v!r}")

✅ 114 asset(s)
\nCampos do 1º asset:
  ID: 411
  IDParent: 0
  SyncPosition: 0
  SyncOperation: ''
  NodeType: 0
  Name: 'VENTILADOR - 136'
  Description: 'VENTILADOR DA COLUNA FINAL DO SISTEMA DE RECUPERAÇÃO DE GASES'
  Path: 'LDC ALTO ARAGUAIA\\MOAGEM\\EXTRAÇÃO\\VENTILADOR - 136'
  Status: [2, 9]
  StatusChanged: '2026-07-27T08:26:24.877'
  Acknowledged: True


In [15]:
# ── build_asset_index() — linha 294 ──────────────────────────
# EDITE AQUI para ajustar a lógica de parse do Path
def build_asset_index(assets_list: list) -> dict:
    """
    Hierarquia (linhas 299-316 do app):
      5+ partes → Unidade / Área / Setor / Equipamento / Ativo
      4  partes → Área='Moagem'? sim → Unidade/Área/Setor/Equip | não → Unidade/Setor/Equip/Ativo
      3  partes → Unidade / Setor / Equipamento
    """
    def _is_area(text: str) -> bool:
        return text.strip().lower() == "moagem"      # linha 321

    index = {}
    for a in assets_list:
        mid  = a.get("ID") or a.get("id")
        if mid is None:
            continue
        path = a.get("Path") or a.get("path") or ""
        desc = a.get("Description") or a.get("description") or ""
        parts = [p.strip() for p in re.split(r"[/\\\\>|]", path) if p.strip()]
        n = len(parts)

        unidade = parts[0] if n > 0 else "—"

        if n >= 5:                                   # linha 337
            area, setor = parts[1], parts[2]
            equipamento, ativo = parts[3], parts[4]
        elif n == 4:                                 # linha 345
            if _is_area(parts[1]):
                area, setor = parts[1], parts[2]
                equipamento = parts[3]
                ativo = desc.strip() or "—"
            else:
                area = "—"
                setor, equipamento, ativo = parts[1], parts[2], parts[3]
        elif n == 3:                                 # linha 360
            area = "—"
            setor, equipamento = parts[1], parts[2]
            ativo = desc.strip() or "—"
        else:
            area = "—"
            setor = parts[1] if n > 1 else "—"
            equipamento = "—"
            ativo = desc.strip() or "—"

        index[int(mid)] = {
            "Unidade": unidade, "Area": area, "Setor": setor,
            "Equipamento": equipamento, "Ativo": ativo,
            "PathCompleto": path, "Descricao": desc,
        }
    return index

asset_index = build_asset_index(assets)
print(f"✅ {len(asset_index)} asset(s) indexados")

✅ 114 asset(s) indexados


In [16]:
# ── Validação do parser de Path ──────────────────────────────
# Mostra os primeiros 20 assets com a hierarquia extraída
rows_check = []
for a in assets[:20]:
    mid  = a.get("ID") or a.get("id")
    info = asset_index.get(int(mid), {})
    rows_check.append({
        "ID":          mid,
        "PathCompleto": info.get("PathCompleto",""),
        "Unidade":     info.get("Unidade",""),
        "Area":        info.get("Area",""),
        "Setor":       info.get("Setor",""),
        "Equipamento": info.get("Equipamento",""),
        "Ativo":       info.get("Ativo",""),
    })

df_check = pd.DataFrame(rows_check)
print("Verificação do parse de Path (primeiros 20 assets):")
df_check

Verificação do parse de Path (primeiros 20 assets):


,ID,PathCompleto,Unidade,Area,Setor,Equipamento,Ativo
0,411,LDC ALTO ARAGUAIA\MOAGEM\EXTRAÇÃO\VENTILADOR -...,LDC ALTO ARAGUAIA,MOAGEM,EXTRAÇÃO,VENTILADOR - 136,VENTILADOR DA COLUNA FINAL DO SISTEMA DE RECUP...
1,457,LDC ALTO ARAGUAIA\MOAGEM\EXTRAÇÃO\13SR,LDC ALTO ARAGUAIA,MOAGEM,EXTRAÇÃO,13SR,SECADOR DE FARELO
2,532,LDC ALTO ARAGUAIA\MOAGEM\EXTRAÇÃO\ M 36B,LDC ALTO ARAGUAIA,MOAGEM,EXTRAÇÃO,M 36B,EXAUSTOR DC
3,610,LDC ALTO ARAGUAIA\MOAGEM\EXTRAÇÃO\3T,LDC ALTO ARAGUAIA,MOAGEM,EXTRAÇÃO,3T,EXTRATOR
4,653,LDC ALTO ARAGUAIA\MOAGEM\EXTRAÇÃO\70D - Aciona...,LDC ALTO ARAGUAIA,MOAGEM,EXTRAÇÃO,70D - Acionamento DTDC,ACIONAMENTO DO DESSOLVENTIZADOR TOSTADOR
5,719,LDC ALTO ARAGUAIA\MOAGEM\EXTRAÇÃO\M8A,LDC ALTO ARAGUAIA,MOAGEM,EXTRAÇÃO,M8A,ROSCA ALIMENTAÇÃO EXTRATOR
6,773,LDC ALTO ARAGUAIA\MOAGEM\EXTRAÇÃO\8B1,LDC ALTO ARAGUAIA,MOAGEM,EXTRAÇÃO,8B1,VÁLVULA ROTATIVA ALIMENTAÇÃO DT
7,827,LDC ALTO ARAGUAIA\MOAGEM\EXTRAÇÃO\8B2,LDC ALTO ARAGUAIA,MOAGEM,EXTRAÇÃO,8B2,VÁLVULA ROTATIVA ALIMENTAÇÃO DO DC
8,860,LDC ALTO ARAGUAIA\MOAGEM\EXTRAÇÃO\8C1,LDC ALTO ARAGUAIA,MOAGEM,EXTRAÇÃO,8C1,VÁLVULA ROTATIVA DESCARGA 13SR
9,893,LDC ALTO ARAGUAIA\MOAGEM\EXTRAÇÃO\8C2,LDC ALTO ARAGUAIA,MOAGEM,EXTRAÇÃO,8C2,VÁLVULA ROTATIVA 2° PISO DESCARGA 13SR


# ─────────────────────────────────────────────────────────────
# 4. POINTS
# ─────────────────────────────────────────────────────────────

In [17]:
# ── Seleciona um asset para explorar ─────────────────────────
df_assets = pd.DataFrame([{
    "ID": a.get("ID"), "Nome": a.get("Name"),
    "Path": a.get("Path"), "Status": a.get("Status")
} for a in assets])

ASSET_IDX  = 0                          # ← índice na lista de assets
ASSET_ID   = df_assets.iloc[ASSET_IDX]["ID"]
ASSET_NAME = df_assets.iloc[ASSET_IDX]["Nome"]
print(f"Asset selecionado: [{ASSET_ID}] {ASSET_NAME}")
print(f"Path: {df_assets.iloc[ASSET_IDX]['Path']}")


Asset selecionado: [411] VENTILADOR - 136
Path: LDC ALTO ARAGUAIA\MOAGEM\EXTRAÇÃO\VENTILADOR - 136


In [18]:
# ── get_points() /v2/points — linha 385 ──────────────────────
def get_points(machine_id):
    resp = requests.get(
        f"{BASE_URL}/v2/points",
        headers=hdrs(),
        params={"machine_id": machine_id},
        timeout=TIMEOUT,
    )
    resp.raise_for_status()
    data = resp.json()
    return data if isinstance(data, list) else data.get("value", [])

pts_v2 = get_points(ASSET_ID)
df_pts_v2 = pd.DataFrame([{
    "ID":       p.get("id") or p.get("ID"),
    "Nome":     p.get("name") or p.get("Name"),
    "NodeType": p.get("NodeType") or p.get("nodeType"),
    "EUType":   p.get("EUType")   or p.get("euType"),
    "ParentID": p.get("ParentID") or p.get("parentId"),
    "Unit":     p.get("unit")     or p.get("Unit"),
} for p in pts_v2])

print(f"✅ /v2/points: {len(df_pts_v2)} point(s)")
df_pts_v2

HTTPError: 404 Client Error: Not Found for url: http://services.repcenter.skf.com:21221/v2/points?machine_id=411

In [ ]:
# ── get_points_v1() /v1/machines/{id}/points — linha 755 ─────
# Retorna NodeType — necessário para filtrar IMx-1
def get_points_v1(machine_id):
    resp = requests.get(
        f"{BASE_URL}/v1/machines/{machine_id}/points",
        headers=hdrs(),
        timeout=15,
    )
    if resp.status_code in (204, 404):
        return []
    resp.raise_for_status()
    data = resp.json()
    return data if isinstance(data, list) else []

pts_v1 = get_points_v1(ASSET_ID)
df_pts_v1 = pd.DataFrame([{
    "ID":       p.get("ID") or p.get("id"),
    "Nome":     p.get("Name") or p.get("name"),
    "NodeType": p.get("NodeType"),
    "EUType":   p.get("EUType"),
    "ParentID": p.get("ParentID") or p.get("IDNode"),
} for p in pts_v1])

print(f"✅ /v1/machines/points: {len(df_pts_v1)} point(s)")
print(f"   IMx-1 (NodeType 11101-11104): {df_pts_v1[df_pts_v1['NodeType'].isin(IMX1_NODE_TYPES)].shape[0]}")
df_pts_v1

In [ ]:
# ── Seleciona um point para testar tendência e espectro ───────
POINT_IDX = 0
POINT_ID  = df_pts_v2.iloc[POINT_IDX]["ID"]
POINT_NAME= df_pts_v2.iloc[POINT_IDX]["Nome"]
print(f"Point selecionado: [{POINT_ID}] {POINT_NAME}")
"""))


# ─────────────────────────────────────────────────────────────
# 5. TENDÊNCIA
# ─────────────────────────────────────────────────────────────


In [ ]:
# ── get_trend() — linha 395 ──────────────────────────────────
FROM_DATE    = "2026-01-01T00:00:00Z"
TO_DATE      = "2026-03-31T23:59:59Z"
MAX_READINGS = 500

def get_trend(point_id, from_date=FROM_DATE, to_date=TO_DATE, max_readings=MAX_READINGS):
    resp = requests.get(
        f"{BASE_URL}/v1/points/{point_id}/trendMeasurements",
        headers=hdrs(),
        params={
            "pointId":             point_id,
            "maxNumberOfReadings": max_readings,
            "fromDateUTC":         from_date,
            "toDateUTC":           to_date,
            "descending":          "false",
        },
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json() if resp.status_code != 204 else []

raw_trend = get_trend(POINT_ID)
print(f"✅ {len(raw_trend)} leitura(s) retornada(s)")

if raw_trend:
    print("\\nEstrutura da 1ª leitura:")
    first = raw_trend[0]
    for k, v in first.items():
        if k != "Measurements":
            print(f"  {k}: {v!r}")
    if "Measurements" in first:
        print(f"  Measurements[0]: {first['Measurements'][0]}")

In [ ]:
# ── parse_trend() — linhas 420-513 ───────────────────────────
# EDITE AQUI para ajustar o parser
def parse_trend(data) -> pd.DataFrame:
    rows = []
    for item in data:
        # Formato B — Measurements[] (linha 446)
        if "Measurements" in item and isinstance(item["Measurements"], list):
            ts_raw  = (item.get("ReadingTimeUTC") or item.get("ReadingTime")
                       or item.get("dateUTC") or item.get("timestamp"))
            speed   = item.get("Speed")
            process = item.get("Process")
            for meas in item["Measurements"]:
                rows.append({
                    "timestamp":    ts_raw,
                    "value":        meas.get("Level"),
                    "unit":         meas.get("Units", ""),
                    "channel":      meas.get("Channel", 1),
                    "channel_name": meas.get("ChannelName", "Overall"),
                    "direction":    meas.get("Direction", ""),
                    "bov":          meas.get("BOV"),
                    "speed":        speed,
                    "process":      process,
                })
        # Formato A — campos planos legado (linha 473)
        else:
            ts_raw  = (item.get("timestamp") or item.get("dateUTC")
                       or item.get("date") or item.get("ReadingTimeUTC"))
            val_obj = item.get("value") or {}
            value   = val_obj.get("value") if isinstance(val_obj, dict) else val_obj
            unit    = val_obj.get("unit","") if isinstance(val_obj, dict) else item.get("unit","")
            rows.append({
                "timestamp": ts_raw, "value": value, "unit": unit,
                "channel": 1, "channel_name": item.get("source","Overall"),
                "direction": "", "bov": None, "speed": None, "process": None,
            })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df["value"]     = pd.to_numeric(df["value"],  errors="coerce")
    df["speed"]     = pd.to_numeric(df.get("speed"),   errors="coerce")
    df["process"]   = pd.to_numeric(df.get("process"), errors="coerce")
    df.dropna(subset=["timestamp","value"], inplace=True)
    df.sort_values("timestamp", inplace=True)
    df.drop_duplicates(subset=["timestamp","channel"], keep="last", inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

df_trend = parse_trend(raw_trend)
print(f"✅ {len(df_trend)} leituras | {df_trend['channel'].nunique()} canal(is)")
print(f"   Período : {df_trend['timestamp'].min()} → {df_trend['timestamp'].max()}")
print(f"   Unidade : {df_trend['unit'].iloc[0] if len(df_trend) else '—'}")
df_trend.head()

In [ ]:
# ── get_channel_options() — linha 694 ───────────────────────
def get_channel_options(df: pd.DataFrame) -> dict:
    if df.empty or "channel" not in df.columns:
        return {}
    channels = (df[["channel","channel_name","direction","unit"]]
                .drop_duplicates("channel").sort_values("channel"))
    opts = {}
    for _, row in channels.iterrows():
        label = f"Ch{int(row['channel'])}  {row['channel_name']}"
        if row["direction"]:
            label += f"  [{row['direction']}]"
        label += f"  · {row['unit']}"
        opts[label] = int(row["channel"])
    return opts

ch_opts = get_channel_options(df_trend)
print("Canais disponíveis:")
for lbl, ch in ch_opts.items():
    print(f"  {ch}: {lbl}")

# Seleciona o canal para plotar
CHANNEL = list(ch_opts.values())[0]
df_ch   = df_trend[df_trend["channel"] == CHANNEL].copy()
unit    = df_ch["unit"].iloc[0]
print(f"\\nPlotando canal {CHANNEL} [{unit}] — {len(df_ch)} pontos")

In [ ]:
# ── Plot de Tendência (espelho do app) ────────────────────────
mu    = df_ch["value"].mean()
sigma = df_ch["value"].std()

fig = go.Figure()

# Área preenchida
fig.add_trace(go.Scatter(
    x=df_ch["timestamp"], y=df_ch["value"],
    fill="tozeroy", fillcolor="rgba(50,85,110,0.08)",
    line=dict(color="rgba(0,0,0,0)"), showlegend=False, hoverinfo="skip",
))
# Linha principal
fig.add_trace(go.Scatter(
    x=df_ch["timestamp"], y=df_ch["value"],
    mode="lines", name=f"{list(ch_opts.keys())[0]}",
    line=dict(color=C_ACCENT, width=1.8),
    hovertemplate="%{x|%d/%m %H:%M}<br><b>%{y:.4f}</b> " + unit + "<extra></extra>",
))
# Banda ±1σ
fig.add_trace(go.Scatter(
    x=list(df_ch["timestamp"]) + list(df_ch["timestamp"][::-1]),
    y=[mu+sigma]*len(df_ch) + [mu-sigma]*len(df_ch),
    fill="toself", fillcolor="rgba(50,85,110,0.10)",
    line=dict(color="rgba(0,0,0,0)"), name="±1σ",
))
# Linhas de referência
fig.add_hline(y=mu,             line=dict(color=C_OK,     width=1,   dash="dash"),
              annotation_text=f"μ={mu:.4f}", annotation_font=dict(color=C_OK, size=9))
fig.add_hline(y=mu+2*sigma,     line=dict(color=C_WARN,   width=1,   dash="dot"),
              annotation_text="Alerta (μ+2σ)", annotation_font=dict(color=C_WARN, size=9))
fig.add_hline(y=mu+3*sigma,     line=dict(color=C_DANGER, width=1.2, dash="dot"),
              annotation_text="Alarme (μ+3σ)", annotation_font=dict(color=C_DANGER, size=9))
# RPM eixo secundário
df_spd = df_ch.dropna(subset=["speed"])
if not df_spd.empty and df_spd["speed"].max() > 0:
    fig.add_trace(go.Scatter(
        x=df_spd["timestamp"], y=df_spd["speed"], mode="lines",
        name="RPM", yaxis="y2",
        line=dict(color=C_PURPLE, width=1.2, dash="dot"),
    ))
    fig.update_layout(yaxis2=dict(
        title=dict(text="RPM", font=dict(color=C_PURPLE, size=10)),
        overlaying="y", side="right", gridcolor="rgba(0,0,0,0)",
        tickfont=dict(size=9, color=C_PURPLE),
    ))

layout = {**PLOTLY_BASE}
layout.update(dict(
    title=dict(text=f"<b>Tendência</b> · {POINT_NAME} · Canal {CHANNEL} [{unit}]",
               font=dict(size=16, color=C_TEXT), x=0.01),
    xaxis=dict(**PLOTLY_BASE["xaxis"],
               title=dict(text="Data/Hora", font=dict(color=C_TEXT_MID))),
    yaxis=dict(**PLOTLY_BASE["yaxis"],
               title=dict(text=f"Nível [{unit}]", font=dict(color=C_TEXT_MID))),
    height=440,
))
fig.update_layout(**layout)
fig.show()
print(f"  μ={mu:.4f}  σ={sigma:.4f}  min={df_ch['value'].min():.4f}  max={df_ch['value'].max():.4f}")

# ─────────────────────────────────────────────────────────────
# 6. ESPECTRO
# ─────────────────────────────────────────────────────────────

In [ ]:
# ── get_spectrum() — linha 410 ───────────────────────────────
def get_spectrum(point_id):
    resp = requests.get(
        f"{BASE_URL}/v1/points/{point_id}/dynamicMeasurements",
        headers=hdrs(), timeout=30,
    )
    if resp.status_code in (204, 404):
        return None
    resp.raise_for_status()
    return resp.json()

raw_spec = get_spectrum(POINT_ID)
if raw_spec:
    item = raw_spec if isinstance(raw_spec, dict) else raw_spec[0]
    print("Metadados do espectro:")
    for k, v in item.items():
        if k != "Measurements":
            print(f"  {k}: {v!r}")
    meas = item.get("Measurements", [])
    print(f"\\n  Measurements: {len(meas)} canal(is)")
    for i, m in enumerate(meas):
        vals = m.get("Values", [])
        print(f"    Canal {i}: Direction={m.get('Direction')}  "
              f"MeasType={m.get('MeasurementType')}  len(Values)={len(vals)}")
else:
    print("⚠ Ponto sem medição dinâmica (espectro)")

In [ ]:
# ── parse_spectrum() — linhas 515-596 ───────────────────────
# EDITE AQUI para ajustar o parser
DIRECTION_MAP = {0:"X", 1:"Y", 2:"Z", 3:"H", 4:"V", 5:"A"}
MTYPE_MAP     = {0:"Waveform", 1:"Espectro (vel.)", 2:"Espectro", 3:"Cepstrum"}

def parse_spectrum(data):
    if data is None:
        return []
    items = data if isinstance(data, list) else [data]
    channels = []
    for item in items:
        eu          = item.get("EU") or item.get("EUSpectrum") or "—"
        start_f     = float(item.get("StartFrequency", 0.0))
        end_f       = float(item.get("EndFrequency",   0.0))
        sample_rate = float(item.get("SampleRate",     0.0))
        samples     = int(item.get("Samples",          0))
        speed       = item.get("Speed", 0.0) or 0.0
        speed_units = item.get("SpeedUnits", "RPM")
        for idx, meas in enumerate(item.get("Measurements") or []):
            values = meas.get("Values") or []
            if not values:
                continue
            n     = len(values)
            freqs = np.linspace(start_f, end_f, n) if end_f > start_f else np.arange(n)
            channels.append({
                "channel_idx": idx,
                "direction":   meas.get("Direction", idx),
                "mtype":       meas.get("MeasurementType", 0),
                "eu": eu, "start_freq": start_f, "end_freq": end_f,
                "freqs":  np.array(freqs, dtype=float),
                "values": np.array(values, dtype=float),
                "sample_rate": sample_rate, "samples": samples,
                "speed": float(speed), "speed_units": speed_units,
            })
    return channels

chs = parse_spectrum(raw_spec)
print(f"✅ {len(chs)} canal(is) parseado(s)")
for ch in chs:
    d = DIRECTION_MAP.get(ch["direction"], str(ch["direction"]))
    m = MTYPE_MAP.get(ch["mtype"], f"Tipo {ch['mtype']}")
    print(f"  Canal {ch['channel_idx']}: dir={d}  tipo={m}  eu={ch['eu']}  "
          f"n={len(ch['values'])}  f={ch['start_freq']:.0f}-{ch['end_freq']:.0f} Hz")

In [ ]:
# ── Plot do Espectro ─────────────────────────────────────────
if not chs:
    print("⚠ Sem espectro para plotar.")
else:
    CH_IDX = 0         # ← altere para outro canal
    ch     = chs[CH_IDX]
    freqs  = ch["freqs"]
    values = ch["values"]
    eu     = ch["eu"]
    dir_s  = DIRECTION_MAP.get(ch["direction"], str(ch["direction"]))
    speed  = ch["speed"]

    # Top 10 picos
    pk_idx = np.argsort(values)[::-1][:10]
    pk_f, pk_v = freqs[pk_idx], values[pk_idx]

    # Harmônicas de rotação
    harm = []
    if speed > 0:
        for h in range(1, 6):
            hf = (speed / 60) * h
            if ch["start_freq"] <= hf <= ch["end_freq"]:
                harm.append((hf, f"{h}X {hf:.1f}Hz"))

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=freqs, y=values, fill="tozeroy",
                             fillcolor="rgba(50,85,110,0.08)",
                             line=dict(color="rgba(0,0,0,0)"),
                             showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=freqs, y=values, mode="lines",
                             name=f"Espectro [{dir_s}]",
                             line=dict(color=C_ACCENT, width=1.5),
                             hovertemplate="<b>%{x:.2f} Hz</b><br>%{y:.5f} " + eu + "<extra></extra>"))
    fig.add_trace(go.Scatter(x=pk_f, y=pk_v, mode="markers+text",
                             name="Picos",
                             marker=dict(size=8, color=C_WARN, symbol="triangle-up"),
                             text=[f"{f:.1f}" for f in pk_f],
                             textposition="top center",
                             textfont=dict(size=8, color=C_WARN),
                             hovertemplate="<b>%{x:.2f} Hz</b><br>%{y:.5f} "+eu+"<extra>Pico</extra>"))
    for hf, hlabel in harm:
        fig.add_vline(x=hf, line=dict(color="rgba(94,105,158,0.6)", width=1, dash="dot"),
                      annotation_text=hlabel, annotation_font=dict(size=8, color=C_PURPLE))

    layout = {**PLOTLY_BASE}
    layout.update(dict(
        title=dict(text=f"<b>Espectro FFT</b> · {POINT_NAME} · Dir {dir_s} [{eu}]",
                   font=dict(size=16, color=C_TEXT), x=0.01),
        xaxis=dict(**PLOTLY_BASE["xaxis"],
                   title=dict(text="Frequência [Hz]", font=dict(color=C_TEXT_MID)),
                   rangeslider=dict(visible=True, bgcolor=C_PANEL, thickness=0.05)),
        yaxis=dict(**PLOTLY_BASE["yaxis"],
                   title=dict(text=f"Amplitude [{eu}]", font=dict(color=C_TEXT_MID))),
        height=460,
    ))
    fig.update_layout(**layout)
    fig.show()
    print(f"  Pico máx: {pk_f[0]:.2f} Hz → {pk_v[0]:.5f} {eu}")

# ─────────────────────────────────────────────────────────────
# 7. IMx-1
# ─────────────────────────────────────────────────────────────

In [ ]:
# ── get_imx_sensors() — linha 795 ────────────────────────────
# Filtra apenas sensores comissionados (Commissioned == 1)
def get_imx_sensors():
    resp = requests.get(f"{BASE_URL}/v1/nextgensensor",
                        headers=hdrs(), timeout=15)
    if resp.status_code in (204, 404):
        return {}
    resp.raise_for_status()
    data    = resp.json()
    sensors = data if isinstance(data, list) else data.get("value", data.get("items", []))
    index   = {}
    for s in (sensors if isinstance(sensors, list) else []):
        comm = (s.get("Commissioned") if s.get("Commissioned") is not None
                else s.get("commissioned"))
        if not comm:
            continue
        nid = s.get("IDNode") or s.get("idNode") or s.get("NodeID")
        if nid is not None:
            index[int(nid)] = s
    return index

sensor_index = get_imx_sensors()
total_raw    = len(get_imx_sensors.__wrapped__().__class__) if False else "N/A"
print(f"✅ {len(sensor_index)} sensor(es) comissionado(s) indexados por IDNode")

# Inspeciona campos disponíveis no 1º sensor
if sensor_index:
    sample = next(iter(sensor_index.values()))
    print("\\nCampos disponíveis no nextgensensor:")
    for k, v in sample.items():
        print(f"  {k}: {v!r}")

In [ ]:
# ── get_trend_first_reading() — linha 769 ────────────────────
def get_trend_first_reading(point_id):
    resp = requests.get(
        f"{BASE_URL}/v1/points/{point_id}/trendMeasurements",
        headers=hdrs(),
        params={"fromDateUTC": IMX1_FROM_DATE, "descending": "false", "numReadings": 1},
        timeout=15,
    )
    if resp.status_code in (204, 404):
        return None
    resp.raise_for_status()
    data = resp.json()
    if isinstance(data, list) and data:
        return data[0]
    if isinstance(data, dict):
        return data
    return None

# Teste rápido com o point selecionado
first = get_trend_first_reading(POINT_ID)
if first:
    ts = (first.get("ReadingTimeUTC") or first.get("readingTimeUTC")
          or first.get("timestamp") or first.get("dateUTC"))
    print(f"✅ 1ª leitura: {ts}")
    print(f"   Campos: {list(first.keys())}")
else:
    print("⚠ Sem leitura a partir de 01/05/2024 para este point")

In [ ]:
# ── Varredura IMx de um único asset (teste rápido) ────────────
# Para rodar a varredura completa use run_imx_scan() do app
print(f"Varrendo [{ASSET_ID}] {ASSET_NAME}…")

pts_v1   = get_points_v1(ASSET_ID)
imx_pts  = [p for p in pts_v1 if int(p.get("NodeType", 0)) in IMX1_NODE_TYPES]
print(f"  Points IMx-1: {len(imx_pts)}")

# Agrupa por IDNode (sensor físico)
nodes: dict[int, list] = {}
for p in imx_pts:
    nid = p.get("ParentID") or p.get("IDNode") or p.get("NodeID")
    try:    nid = int(nid)
    except: nid = None
    if nid:
        nodes.setdefault(nid, []).append(p)

print(f"  IDNodes: {list(nodes.keys())}")
today = datetime.now(timezone.utc)
rows_imx = []

for id_node, node_pts in nodes.items():
    if id_node not in sensor_index:
        print(f"  ⊘ IDNode {id_node} — não comissionado")
        continue

    # Melhor point (temperatura > outro) — linha 929
    temp_pts    = [p for p in node_pts if int(p.get("NodeType",0)) == IMX1_TEMP_NODE_TYPE
                   or int(p.get("EUType",0)) == IMX1_TEMP_EU_TYPE]
    target      = temp_pts[0] if temp_pts else node_pts[0]
    pid         = target.get("ID") or target.get("id")
    point_name  = target.get("Name") or target.get("name") or "—"

    first = get_trend_first_reading(pid)
    time.sleep(0.15)

    ts_raw = (first.get("ReadingTimeUTC") or first.get("readingTimeUTC")
              if first else None)
    try:    commissioning_dt = pd.to_datetime(ts_raw, utc=True)
    except: commissioning_dt = None

    meta    = sensor_index.get(id_node, {})
    hw_id   = meta.get("SensorIdentifier") or "—"
    battery = meta.get("BatteryLevel")
    try:    battery = float(battery)
    except: battery = None

    # ClearedDate — sentinela ano ≤ 1940 (linha 986)
    cleared_raw = (meta.get("ClearedDate") or meta.get("clearedDate")
                   or meta.get("Cleared"))
    try:    cleared_dt = pd.to_datetime(cleared_raw, utc=True) if cleared_raw else None
    except: cleared_dt = None
    if cleared_dt and cleared_dt.year <= 1940:
        print(f"  ⚠ IDNode {id_node}: ClearedDate sentinela ({cleared_dt.year}) — ignorado")
        cleared_dt = None

    effective_dt = cleared_dt or commissioning_dt
    fonte        = "ClearedDate" if cleared_dt else ("1ª Leitura" if commissioning_dt else "—")
    dias_uso     = (today - effective_dt).days if effective_dt else None
    taxa_bat     = round((100 - battery) / dias_uso, 4) \
                   if dias_uso and dias_uso > 0 and battery is not None else None

    loc = asset_index.get(int(ASSET_ID), {})

    rows_imx.append({
        "HardwareID":    hw_id,
        "IDNode":        id_node,
        "MachineID":     ASSET_ID,
        "MachineName":   ASSET_NAME,
        "NomePonto":     point_name,
        "Unidade":       loc.get("Unidade","—"),
        "Area":          loc.get("Area","—"),
        "Setor":         loc.get("Setor","—"),
        "Equipamento":   loc.get("Equipamento","—"),
        "Ativo":         loc.get("Ativo","—"),
        "BatteryLevel":  battery,
        "DataPrimeiraLeitura": commissioning_dt.strftime("%Y-%m-%d %H:%M") if commissioning_dt else None,
        "DataClearedSensor":   cleared_dt.strftime("%Y-%m-%d %H:%M") if cleared_dt else None,
        "ProvavelDataComissionamento": effective_dt.strftime("%Y-%m-%d %H:%M") if effective_dt else None,
        "FonteComissionamento": fonte,
        "DiasDeUso":     dias_uso,
        "TaxaConsumoBateria": taxa_bat,
    })
    print(f"  ✓ IDNode {id_node} | HW: {hw_id} | "
          f"Bat: {battery}% | Ponto: {point_name} | Comissionado: {effective_dt} ({fonte})")

df_imx = pd.DataFrame(rows_imx)
print(f"\\n✅ DataFrame IMx: {len(df_imx)} linha(s)")
df_imx

In [ ]:
# ── Plot: Bateria × Dias em Campo ────────────────────────────
if len(df_imx) >= 1:
    df_p = df_imx.dropna(subset=["BatteryLevel", "DiasDeUso"]).copy()
    if df_p.empty:
        print("Sem dados suficientes para o scatter.")
    else:
        def bat_color(b):
            return C_DANGER if b < 20 else (C_WARN if b < 40 else C_OK)

        fig = go.Figure()
        fig.add_hrect(y0=0,  y1=20,  fillcolor="rgba(240,106,34,0.07)", line_width=0)
        fig.add_hrect(y0=20, y1=40,  fillcolor="rgba(186,148,75,0.05)", line_width=0)
        fig.add_hrect(y0=40, y1=100, fillcolor="rgba(78,157,45,0.03)",  line_width=0)
        fig.add_trace(go.Scatter(
            x=df_p["DiasDeUso"], y=df_p["BatteryLevel"], mode="markers",
            marker=dict(size=11, color=[bat_color(b) for b in df_p["BatteryLevel"]],
                        line=dict(color="rgba(255,255,255,0.15)", width=1)),
            text=df_p["MachineName"],
            customdata=df_p[["HardwareID","NomePonto","TaxaConsumoBateria"]].values,
            hovertemplate=(
                "<b>%{text}</b><br>HW: %{customdata[0]}<br>"
                "Ponto: %{customdata[1]}<br>"
                "Bateria: %{y:.1f}%<br>Dias: %{x}<br>"
                "Taxa: %{customdata[2]:.4f}%/dia<extra></extra>"
            ),
        ))
        layout = {**PLOTLY_BASE}
        layout.update(dict(
            title=dict(text="<b>Bateria × Dias em Campo</b>",
                       font=dict(size=16, color=C_TEXT), x=0.01),
            xaxis=dict(**PLOTLY_BASE["xaxis"],
                       title=dict(text="Dias em Campo", font=dict(color=C_TEXT_MID))),
            yaxis=dict(**PLOTLY_BASE["yaxis"], range=[0,105],
                       title=dict(text="Bateria [%]", font=dict(color=C_TEXT_MID))),
            height=380,
        ))
        fig.update_layout(**layout)
        fig.show()

# ─────────────────────────────────────────────────────────────
# 8. FLEET
# ─────────────────────────────────────────────────────────────


In [ ]:
# ── get_gateways() — linha 1073 ──────────────────────────────
def get_gateways():
    resp = requests.get(f"{BASE_URL}/v1/gateways",
                        headers=hdrs(), timeout=15)
    if resp.status_code in (204, 404): return []
    resp.raise_for_status()
    data = resp.json()
    return data if isinstance(data, list) else data.get("value", [])

def _parse_dt(raw):
    if not raw: return None
    try:    return pd.to_datetime(raw, utc=True)
    except: return None

today_utc = datetime.now(timezone.utc)
gw_raw    = get_gateways()
gw_rows   = []
for g in gw_raw:
    updated   = _parse_dt(g.get("statusLastUpdated") or g.get("StatusLastUpdated"))
    connected = g.get("connected") if g.get("connected") is not None else g.get("Connected")
    gw_rows.append({
        "GatewayID":         g.get("id") or g.get("ID"),
        "Nome":              g.get("name") or g.get("Name") or "—",
        "Conectado":         bool(connected),
        "StatusLastUpdated": updated.strftime("%Y-%m-%d %H:%M:%S") if updated else None,
        "DiasSemUpdate":     (today_utc - updated).days if updated else None,
        "Status":            "🟢 Online" if connected else "🔴 Offline",
    })

df_gw = pd.DataFrame(gw_rows)
print(f"✅ {len(df_gw)} gateway(s) | "
      f"Online: {df_gw['Conectado'].sum()} | "
      f"Offline: {(~df_gw['Conectado']).sum()}")
df_gw

In [ ]:
# ── Sensores /v1/nextgensensor com localização ───────────────
# Replica run_fleet_scan() linhas 1228-1320
# node_to_loc: construído via /v2/points por asset (linhas 1153-1201)
print("Construindo índice IDNode → localização…")
node_to_loc: dict[int, dict] = {}
for a in assets:
    mid = a.get("ID") or a.get("id")
    if mid is None: continue
    mid_int = int(mid)
    if mid_int not in asset_index: continue
    try:
        r = requests.get(f"{BASE_URL}/v2/points", headers=hdrs(),
                         params={"machine_id": mid}, timeout=15)
        pts = r.json() if r.status_code == 200 else []
        pts = pts if isinstance(pts, list) else pts.get("value", [])
    except Exception:
        pts = []
    for p in pts:
        nid = p.get("ParentID") or p.get("parentId")
        try:    nid = int(nid)
        except: nid = None
        if nid is not None:
            node_to_loc[nid] = asset_index[mid_int]
    time.sleep(0.05)

print(f"✅ {len(node_to_loc)} IDNode(s) mapeados")

# Coleta sensores
resp_ns  = requests.get(f"{BASE_URL}/v1/nextgensensor", headers=hdrs(), timeout=15)
ns_raw   = resp_ns.json() if resp_ns.status_code == 200 else []
ns_all   = ns_raw if isinstance(ns_raw, list) else ns_raw.get("value", [])
ns_list  = [s for s in ns_all if (s.get("Commissioned") if s.get("Commissioned") is not None
                                   else s.get("commissioned"))]
print(f"✅ {len(ns_list)} comissionado(s) de {len(ns_all)} total")

sensor_rows = []
for s in ns_list:
    updated     = _parse_dt(s.get("StatusLastUpdated") or s.get("statusLastUpdated"))
    dias_off    = (today_utc - updated).days if updated else None
    bat         = s.get("BatteryLevel") or s.get("batteryLevel")
    try:    bat = float(bat)
    except: bat = None

    diag        = s.get("DiagnosticCode") or 0
    try:    diag = int(diag)
    except: diag = 0
    flags       = [lbl for bit, lbl in DIAG_BITS.items() if diag & bit]

    conn_raw    = s.get("ConnectionState") or s.get("connectionState")
    try:    conn_code = int(conn_raw) if conn_raw is not None else None
    except: conn_code = None
    conn_lbl    = CONN_STATE_LABELS.get(conn_code, f"? ({conn_raw})")

    if conn_code == 0 or (dias_off and dias_off > 2):   alert = "danger"
    elif bat is not None and bat < 20:                    alert = "danger"
    elif conn_code in (2, 3):                             alert = "warn"
    elif diag & 512:                                      alert = "warn"
    elif bat is not None and bat < 40:                    alert = "warn"
    else:                                                 alert = "ok"

    id_node = int(s.get("IDNode") or s.get("idNode") or 0)
    loc     = node_to_loc.get(id_node, {})

    sensor_rows.append({
        "IDNode":          id_node,
        "HardwareID":      s.get("SensorIdentifier") or "—",
        "GatewayID":       s.get("IDSmartGateway") or s.get("idSmartGateway"),
        "ConnectionState": conn_lbl,
        "BatteryLevel":    bat,
        "LastUpdate":      updated.strftime("%Y-%m-%d %H:%M:%S") if updated else None,
        "DiasOffline":     dias_off,
        "DiagFlags":       ", ".join(flags) if flags else "—",
        "Alert":           alert,
        "Unidade":         loc.get("Unidade","—"),
        "Area":            loc.get("Area","—"),
        "Setor":           loc.get("Setor","—"),
        "Equipamento":     loc.get("Equipamento","—"),
        "Ativo":           loc.get("Ativo","—"),
    })

df_sens = pd.DataFrame(sensor_rows)
print(f"\\nConectados:    {(df_sens['ConnectionState']=='Conectado').sum()}")
print(f"Desconectados: {(df_sens['ConnectionState']=='Desconectado').sum()}")
print(f"Sem Medição:   {df_sens['ConnectionState'].isin(['Sem Medição','Conectado — Sem Medição']).sum()}")
print(f"Bat < 20%:     {(df_sens['BatteryLevel'].fillna(100) < 20).sum()}")
print(f"Localização preenchida: {(df_sens['Unidade'] != '—').sum()} de {len(df_sens)}")
df_sens.head(10)

In [ ]:
# ── Plot: Scatter Bateria × Dias Offline (replica app) ───────
df_sc = df_sens.dropna(subset=["BatteryLevel"]).copy()
df_sc["DiasOffline"] = df_sc["DiasOffline"].fillna(0)
ALERT_COLORS = {"ok": C_OK, "warn": C_WARN, "danger": C_DANGER}
colors = [ALERT_COLORS.get(a, C_TEAL) for a in df_sc["Alert"]]

fig = go.Figure()
fig.add_hrect(y0=0,  y1=20,  fillcolor="rgba(240,106,34,0.07)", line_width=0)
fig.add_hrect(y0=20, y1=40,  fillcolor="rgba(186,148,75,0.05)", line_width=0)
fig.add_hrect(y0=40, y1=100, fillcolor="rgba(78,157,45,0.03)",  line_width=0)
fig.add_vline(x=2, line=dict(color="rgba(240,106,34,0.4)", width=1, dash="dot"),
              annotation_text="2 dias", annotation_font=dict(color=C_DANGER, size=9))
fig.add_hline(y=20, line=dict(color="rgba(240,106,34,0.4)", width=1, dash="dot"),
              annotation_text="Bat 20%", annotation_font=dict(color=C_DANGER, size=9),
              annotation_position="bottom right")
fig.add_trace(go.Scatter(
    x=df_sc["DiasOffline"], y=df_sc["BatteryLevel"], mode="markers",
    marker=dict(size=11, color=colors, line=dict(color="rgba(255,255,255,0.15)", width=1)),
    customdata=df_sc[["HardwareID","ConnectionState","DiagFlags",
                       "Unidade","Area","Setor","Ativo"]].values,
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Estado: %{customdata[1]}<br>"
        "Bateria: %{y:.1f}%<br>Dias offline: %{x}<br>"
        "Diagnóstico: %{customdata[2]}<br>"
        "📍 %{customdata[3]} › %{customdata[4]} › %{customdata[5]} › %{customdata[6]}"
        "<extra></extra>"
    ),
))
layout = {**PLOTLY_BASE}
layout.update(dict(
    title=dict(text="<b>Saúde da Rede Mesh</b> · Bateria × Dias sem Atualização",
               font=dict(size=16, color=C_TEXT), x=0.01),
    xaxis=dict(**PLOTLY_BASE["xaxis"],
               title=dict(text="Dias sem Atualização", font=dict(color=C_TEXT_MID))),
    yaxis=dict(**PLOTLY_BASE["yaxis"], range=[0,105],
               title=dict(text="Bateria [%]", font=dict(color=C_TEXT_MID))),
    height=420, hovermode="closest",
))
fig.update_layout(**layout)
fig.show()

# ─────────────────────────────────────────────────────────────
# 9. PATH PARSER
# ─────────────────────────────────────────────────────────────

In [ ]:
# ── Testes unitários do parser ────────────────────────────────
# Adicione seus casos reais aqui
test_cases = [
    # (path, descricao_esperada, Unidade, Area, Setor, Equipamento, Ativo)
    ("LDC_ALTO_ARAGUAIA / Moagem / Preparação / 13SR / Redutor",  "",
     "LDC_ALTO_ARAGUAIA", "Moagem",  "Preparação", "13SR", "Redutor"),
    ("LDC_PONTA_GROSSA / Extração / 07SR / Motor",                "",
     "LDC_PONTA_GROSSA",  "—",       "Extração",   "07SR", "Motor"),
    ("LDC_JATAI / Preparação / 22SR",                             "Redutor",
     "LDC_JATAI",         "—",       "Preparação", "22SR", "Redutor"),
    ("LDC_ITUMBIARA / Secagem / Secador1 / 05SR / Ventilador",    "",
     "LDC_ITUMBIARA",     "Secagem", "Secador1",   "05SR", "Ventilador"),
    # ← Adicione seus casos reais abaixo:
]

def _parse_path_test(path, desc=""):
    """Versão isolada de build_asset_index para teste rápido."""
    def _is_area(t): return t.strip().lower() == "moagem"
    parts = [p.strip() for p in re.split(r"[/\\\\>|]", path) if p.strip()]
    n = len(parts)
    u = parts[0] if n > 0 else "—"
    if n >= 5:
        ar, se, eq, at = parts[1], parts[2], parts[3], parts[4]
    elif n == 4:
        if _is_area(parts[1]):
            ar, se, eq, at = parts[1], parts[2], parts[3], desc or "—"
        else:
            ar, se, eq, at = "—", parts[1], parts[2], parts[3]
    elif n == 3:
        ar, se, eq, at = "—", parts[1], parts[2], desc or "—"
    else:
        ar = "—"; se = parts[1] if n > 1 else "—"; eq = "—"; at = desc or "—"
    return (u, ar, se, eq, at)

print(f"{'Status':<6} {'Path':<55} {'Resultado'}")
print("─" * 120)
all_ok = True
for path, desc, exp_u, exp_ar, exp_se, exp_eq, exp_at in test_cases:
    got = _parse_path_test(path, desc)
    exp = (exp_u, exp_ar, exp_se, exp_eq, exp_at)
    ok  = got == exp
    if not ok: all_ok = False
    icon = "✓" if ok else "✗"
    print(f"  {icon}    {path[:55]:<55}")
    if not ok:
        print(f"         esperado : {exp}")
        print(f"         obtido   : {got}")

print()
print("Todos os casos OK ✓" if all_ok else "⚠ Há casos com erro — ajuste a função acima")

In [ ]:
# ── Teste com assets reais da planta ────────────────────────
# Mostra o breakdown de partes por número de níveis
part_counts = {}
for mid, info in asset_index.items():
    path  = info.get("PathCompleto", "")
    n     = len([p for p in re.split(r"[/\\\\>|]", path) if p.strip()])
    part_counts[n] = part_counts.get(n, 0) + 1

print("Distribuição de níveis no Path:")
for n in sorted(part_counts):
    print(f"  {n} partes: {part_counts[n]} asset(s)")

# Mostra exemplos de cada tamanho
print()
for n_target in sorted(part_counts):
    examples = [(mid, info) for mid, info in asset_index.items()
                if len([p for p in re.split(r'[/\\\\>|]', info['PathCompleto']) if p.strip()]) == n_target]
    ex_mid, ex_info = examples[0]
    print(f"── {n_target} partes (ex): {ex_info['PathCompleto']}")
    print(f"   Unidade={ex_info['Unidade']}  Area={ex_info['Area']}  "
          f"Setor={ex_info['Setor']}  Equip={ex_info['Equipamento']}  Ativo={ex_info['Ativo']}")
    print()